# Preprocess

## Imports

In [ ]:
import torch
from torchvision import transforms
import os
import glob
import yaml
import numpy as np
import tifffile
from pathlib import Path
from PIL import Image
from tqdm import tqdm
import multiprocessing
from functools import partial

## Define Paths and RGB size

In [ ]:
REPO_ROOT = Path.cwd().parent

dataset_name = 'spectralwaste_segmentation'     #adjust for custom datasets
hsi_path = REPO_ROOT / 'datasets' / dataset_name / 'hyper'
rgb_path = REPO_ROOT / 'datasets' / dataset_name / 'rgb'

dest_npz_base   = REPO_ROOT / 'datasets' / f'{dataset_name}_npz'
stats_file_path = REPO_ROOT / 'datasets' / f'{dataset_name}_npz' / 'hsi_stats.yaml'

rgb_resize_size = 256

## Define Preprocess Functions

In [ ]:
def find_rgb_files(rgb_folder: Path) -> list:
    rgb_files = sorted(Path(rgb_folder).glob("*.png"))
    if not rgb_files:
        rgb_files = sorted(Path(rgb_folder).glob("*.jpg"))
    return rgb_files


def collect_image_filesets(rgb_folder_path, hsi_folder_path):
    print(f"Collecting file pairs from RGB: {rgb_folder_path} and HSI: {hsi_folder_path}")
    
    rgb_folder = Path(rgb_folder_path)
    hsi_folder = Path(hsi_folder_path)
    
    rgb_files = find_rgb_files(rgb_folder)
    
    if not rgb_files:
        print(f"Warning: No RGB files (.png or .jpg) found in {rgb_folder_path}")
        return []

    image_sets = []
    for rgb_path in rgb_files:
        file_stem = rgb_path.stem
        hsi_path = hsi_folder / f"{file_stem}.tiff"
        
        if hsi_path.exists():
            image_sets.append({
                'rgb_file': rgb_path.name,
                'hsi_file': hsi_path.name,
            })
        else:
            print(f"  - Warning: No matching HSI file found for {rgb_path.name}")
            
    print(f"Found {len(image_sets)} matching RGB/HSI file pairs.")
    return image_sets

def process_single_image_to_npz(file_set, split_name, src_rgb_folder, src_hsi_folder,
                                dest_npz_folder_base, rgb_transform):
    rgb_filename = file_set['rgb_file']
    hsi_filename = file_set['hsi_file']
    
    src_rgb_path = Path(src_rgb_folder) / rgb_filename
    src_hsi_path = Path(src_hsi_folder) / hsi_filename
    
    dest_split_folder = Path(dest_npz_folder_base) / split_name
    dest_split_folder.mkdir(parents=True, exist_ok=True)

    output_stem = Path(rgb_filename).stem
    npz_filepath = dest_split_folder / f"{output_stem}.npz"

    try:
        if not src_rgb_path.exists():
            return f"Warning: RGB file not found {src_rgb_path}, skipping."
        rgb_pil = Image.open(src_rgb_path).convert('RGB')

        if not src_hsi_path.exists():
            return f"Warning: HSI file not found {src_hsi_path}, skipping."
        hsi_data = tifffile.imread(src_hsi_path)
        
        if hsi_data.ndim != 3:
            return f"Warning: HSI data in {hsi_filename} is not 3D."

        rgb_tensor_transformed = rgb_transform(rgb_pil)
        hsi_tensor_raw = torch.from_numpy(hsi_data.astype(np.float32))

        rgb_numpy = rgb_tensor_transformed.numpy()
        hsi_numpy = hsi_tensor_raw.numpy()
        
        np.savez_compressed(npz_filepath, rgb=rgb_numpy, hsi=hsi_numpy)
        
        return None
    except Exception as e:
        return f"Error processing {output_stem}: {e}"


def execute_npz_processing_for_split(image_file_sets, split_name, src_rgb_folder, src_hsi_folder,
                                     dest_npz_folder_base,
                                     rgb_transform, num_processes=1):
    if not image_file_sets:
        print(f"No image file sets for {split_name}. Skipping.")
        return

    dest_split_folder = Path(dest_npz_folder_base) / split_name
    dest_split_folder.mkdir(parents=True, exist_ok=True)
    print(f"Processing {split_name}. Saving pre-processed .npz files to: {dest_split_folder} using {num_processes} processes.")

    worker_partial = partial(process_single_image_to_npz,
                             split_name=split_name,
                             src_rgb_folder=src_rgb_folder,
                             src_hsi_folder=src_hsi_folder,
                             dest_npz_folder_base=dest_npz_folder_base,
                             rgb_transform=rgb_transform)
    
    errors = []
    processed_count = 0

    if num_processes > 1:
        with multiprocessing.Pool(processes=num_processes) as pool:
            results = list(tqdm(pool.imap_unordered(worker_partial, image_file_sets), 
                              total=len(image_file_sets), desc=f"Saving {split_name} (.npz)"))
            for res in results:
                if res is None:
                    processed_count += 1
                else:
                    errors.append(res)
    else:
        print("Running in single-process mode.")
        for file_set in tqdm(image_file_sets, desc=f"Saving {split_name} (.npz)"):
            res = worker_partial(file_set)
            if res is None:
                processed_count += 1
            else:
                errors.append(res)

    if errors:
        print(f"Finished .npz processing for {processed_count} files in {split_name} with {len(errors)} errors.")
        for err in errors[:5]:
            print(f"  - {err}")
    else:
        print(f"Finished .npz processing for {processed_count} files in {split_name}.")
    
    
def calculate_hsi_statistics_from_npz(npz_folder_path):
    npz_files = sorted(list(Path(npz_folder_path).glob('*.npz')))

    if not npz_files:
        print(f"Error: No .npz files found in {npz_folder_path}")
        return None

    spectral_channels = np.load(npz_files[0])['hsi'].shape[0]
    print(f"Detected {spectral_channels} channels from data")

    channel_sums    = np.zeros(spectral_channels, dtype=np.float64)
    channel_sq_sums = np.zeros(spectral_channels, dtype=np.float64)
    pixel_counts    = np.zeros(spectral_channels, dtype=np.int64)

    for file_path in tqdm(npz_files, desc="Calculating HSI stats from npz"):
        try:
            hsi_data = np.load(file_path)['hsi'].astype(np.float64)  # (S, H, W)
            non_zero_mask = (hsi_data != 0)
            channel_sums    += np.sum(hsi_data * non_zero_mask, axis=(1, 2))
            channel_sq_sums += np.sum((hsi_data * non_zero_mask) ** 2, axis=(1, 2))
            pixel_counts    += np.sum(non_zero_mask, axis=(1, 2))
        except Exception as e:
            print(f"Error processing {file_path.name}: {e}")
            continue

    channel_means   = np.divide(channel_sums, pixel_counts, out=np.zeros_like(channel_sums), where=pixel_counts!=0)
    mean_of_squares = np.divide(channel_sq_sums, pixel_counts, out=np.zeros_like(channel_sq_sums), where=pixel_counts!=0)
    channel_vars    = np.clip(mean_of_squares - np.square(channel_means), 0.0, None)
    channel_stds    = np.sqrt(channel_vars)

    print("HSI Statistics calculation complete.")
    return {'means': channel_means, 'stds': channel_stds}

## Create .npz files

This cell processes raw `.tiff` HSI + `.png`/`.jpg` RGB files into `.npz` format.  
If your custom data is **not** in `.tiff` format, skip this cell and create `.npz` files manually:

```python
np.savez(
    "sample_001.npz",
    rgb = rgb_array,   # shape (3, H_rgb, W_rgb),  dtype float32
    hsi = hsi_array,   # shape (S, H_hsi, W_hsi),  dtype float32
)
```

and place them under: `datasets/<dataset_name>_npz/images/{train,val,test}/`

In [ ]:
print("--- Step 1: Defining Transformation Pipelines ---")
rgb_transform = transforms.Compose([
    transforms.Resize((rgb_resize_size, rgb_resize_size), antialias=True),
    transforms.ToTensor(),
])
print("Image transforms ready.")

print("--- Step 2: Processing images to .npz format ---")
for split in ['train', 'val', 'test']:
    src_rgb_folder = rgb_path / split
    src_hsi_folder = hsi_path / split

    if not src_rgb_folder.exists():
        print(f"Source RGB folder not found, skipping: {src_rgb_folder}")
        continue
    if not src_hsi_folder.exists():
        print(f"Source HSI folder not found, skipping: {src_hsi_folder}")
        continue

    image_sets = collect_image_filesets(str(src_rgb_folder), str(src_hsi_folder))

    if image_sets:
        print(f"\n--- Processing split: {split} ---")
        execute_npz_processing_for_split(
            image_file_sets=image_sets,
            split_name=split,
            src_rgb_folder=str(src_rgb_folder),
            src_hsi_folder=str(src_hsi_folder),
            dest_npz_folder_base=str(dest_npz_base / 'images'),
            rgb_transform=rgb_transform,
            num_processes=1       # Adjust this to >1 for parallel processing (e.g., num_processes=4) 
        )

print("npz creation complete.")

## Calculate HSI stats

In [ ]:
print("--- Step 3: Calculating HSI Normalization Statistics from npz ---")
hsi_stats = calculate_hsi_statistics_from_npz(dest_npz_base / 'images' / 'train')

stats_file_path.parent.mkdir(parents=True, exist_ok=True)
try:
    with open(stats_file_path, 'w', encoding='utf-8') as f:
        yaml.dump({
            'standardize': {
                'hsi_global_means': hsi_stats['means'].tolist(),
                'hsi_global_stds': hsi_stats['stds'].tolist()
            }
        }, f, indent=4)
    print(f"HSI normalization statistics saved to: {stats_file_path}")
except Exception as e:
    print(f"Error saving stats file: {e}")
